# Симуляция матричного акселератора в gem5
## Анализ результатов экспериментов

**Конфигурация:** RISC-V SE режим, матрица N×N (float), акселератор на базе DmaVirtDevice  
**Кэш:** PrivateL1PrivateL2 (32KiB L1D, 32KiB L1I, 256KiB L2)  
**Память:** SingleChannelDDR4_2400, 512MiB

In [ ]:
import csv
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

CSV_DIR = "/home/chern/gem5-project/gem5/accel_project/results/csv"

rows = []

with open(f"{CSV_DIR}/results.csv") as f:
    for r in csv.DictReader(f):
        r["matrix_n"] = 64
        r["cache"]    = "no"
        r["simTicks"] = int(r["simTicks"])
        r["simInsts"] = int(r["simInsts"])
        rows.append(r)

with open(f"{CSV_DIR}/results_cache.csv") as f:
    for r in csv.DictReader(f):
        r["matrix_n"] = 64
        r["simTicks"] = int(r["simTicks"])
        r["simInsts"] = int(r["simInsts"])
        rows.append(r)

with open(f"{CSV_DIR}/results_matrix_size.csv") as f:
    for r in csv.DictReader(f):
        r["matrix_n"] = int(r["matrix_n"])
        r["simTicks"] = int(r["simTicks"])
        r["simInsts"] = int(r["simInsts"])
        rows.append(r)

def get(mode, cpu, n, cache, **kwargs):
    for r in rows:
        if (r["mode"] == mode and r["cpu_type"] == cpu
                and r["matrix_n"] == n and r["cache"] == cache):
            if all(r.get(k) == v for k, v in kwargs.items()):
                return r
    return None

print(f"Загружено строк: {len(rows)}")

## Эксперимент 6: Акселератор vs CPU с кэшем L1+L2

Сравниваем CPU и акселератор при наличии кэша. Две проблемы которые пришлось решить:
1. **STATUS регистр кэшировался** → CPU зависал в бесконечном цикле (`tlb.cc` fix)
2. **DMA + Modified кэш-строки** → паника `cache.cc:1225` → исправлено добавлением IOCache

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("Эксперимент 6: CPU vs Акселератор с кэшем (N=64, block=64, float)", fontsize=12)

configs = [
    ("cpu",   "timing", "CPU\nTIMING"),
    ("cpu",   "o3",     "CPU\nO3"),
    ("accel", "timing", "Accel\nTIMING"),
    ("accel", "o3",     "Accel\nO3"),
]
x  = np.arange(len(configs))
w  = 0.35
nc = [get(m, c, 64, "no")["simTicks"]  for m, c, _ in configs]
ca = [get(m, c, 64, "yes")["simTicks"] for m, c, _ in configs]

for ax, panel_configs, title in [
    (ax1, configs[:2], "CPU (без акселератора)"),
    (ax2, configs[2:], "С акселератором"),
]:
    px   = np.arange(len(panel_configs))
    nc_t = [get(m, c, 64, "no")["simTicks"]  for m, c, _ in panel_configs]
    ca_t = [get(m, c, 64, "yes")["simTicks"] for m, c, _ in panel_configs]
    b1 = ax.bar(px - w/2, [t/1e9 for t in nc_t], w, label="Без кэша",  color="steelblue")
    b2 = ax.bar(px + w/2, [t/1e9 for t in ca_t], w, label="L1+L2 кэш", color="darkorange")
    for bar, t in zip(b1, nc_t):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02*max(nc_t)/1e9,
                f"{t/1e9:.1f}", ha="center", va="bottom", fontsize=9)
    for bar, t in zip(b2, ca_t):
        v = t/1e9
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02*max(nc_t)/1e9,
                f"{v:.3f}" if v<1 else f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    for i, (nt, ct) in enumerate(zip(nc_t, ca_t)):
        ax.text(px[i], max(nt,ct)/1e9*1.08, f"{nt/ct:.1f}×",
                ha="center", va="bottom", fontsize=11, fontweight="bold", color="darkgreen")
    ax.set_xticks(px)
    ax.set_xticklabels([l for _,_,l in panel_configs])
    ax.set_ylabel("simTicks (×10⁹)")
    ax.set_title(title)
    ax.legend(); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print("Ускорение акселератора vs CPU (с кэшем):")
print(f"  TIMING: {get('cpu','timing',64,'yes')['simTicks']/get('accel','timing',64,'yes')['simTicks']:.1f}×")
print(f"  O3:     {get('cpu','o3',64,'yes')['simTicks']/get('accel','o3',64,'yes')['simTicks']:.1f}×")

## Эксперимент 7: Масштабирование по размеру матрицы (N=64/128/256/512)

Проверяем как меняется преимущество акселератора при росте задачи.  
**Гипотеза:** при больших матрицах кэш перестаёт помогать CPU, а акселератор остаётся эффективным.

In [ ]:
ns = [64, 128, 256, 512]
cpu_types = [
    ("timing", "TIMING", "steelblue",  "o"),
    ("o3",     "O3",     "darkorange", "s"),
]

fig, (ax_cache, ax_nocache) = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("Эксперимент 7: Ускорение акселератора vs CPU по размеру матрицы", fontsize=12)

for ax, cache, title in [
    (ax_cache,   "yes", "С кэшем L1+L2"),
    (ax_nocache, "no",  "Без кэша"),
]:
    for cpu, label, color, marker in cpu_types:
        speedups, valid_ns = [], []
        for n in ns:
            cr = get("cpu", cpu, n, cache)
            ar = get("accel", cpu, n, cache)
            if cr and ar:
                speedups.append(cr["simTicks"] / ar["simTicks"])
                valid_ns.append(n)
        if not speedups:
            continue
        ax.plot(valid_ns, speedups, marker=marker, color=color,
                linewidth=2, markersize=8, label=f"vs CPU {label}")
        for n, sp in zip(valid_ns, speedups):
            ax.annotate(f"{sp:.1f}×", (n, sp),
                        textcoords="offset points", xytext=(6, 4), fontsize=9)
    ax.set_xlabel("Размер матрицы N")
    ax.set_ylabel("Ускорение (CPU ticks / Accel ticks)")
    ax.set_xticks(ns)
    ax.set_xticklabels([f"N={n}\n({n}×{n})" for n in ns])
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_title(title)

plt.tight_layout()
plt.show()